# Wikipedia Data Collection Pipeline
Sahana Bai Sankarrao — MATH5872M

## Section 1 : Setup

In [ ]:
!pip install mwxml mwparserfromhell

In [ ]:
import bz2
import re
import time
import requests
import pandas as pd
import mwxml
import mwparserfromhell

CHUNKS = [
    "enwiki-latest-pages-articles-multistream1.xml-p1p41242.bz2",
    "enwiki-latest-pages-articles-multistream2.xml-p41243p151573.bz2",
    "enwiki-latest-pages-articles-multistream3.xml-p151574p311329.bz2",
    "enwiki-latest-pages-articles-multistream4.xml-p311330p558391.bz2",
]
MAX_ARTICLES_PER_CHUNK = 500

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Wikipedia Quality Research; hssl0935@leeds.ac.uk)"
}

## Section 2 : Extract structural features and topic categories from the dump chunks

In [ ]:
def assign_category_from_cats(categories):
    cats_text = " ".join(categories).lower()

    if any(w in cats_text for w in [
        "science", "physics", "chemistry", "biology", "mathematics",
        "astronomy", "geology", "ecology", "medicine", "technology",
        "computing", "software", "engineering", "electronics",
        "species", "animal", "plant", "bird", "insect", "fish",
        "mammal", "bacteria", "virus", "genetics", "evolution",
        "climate", "environment", "quantum", "atomic"
    ]):
        return "Science & Technology"

    elif any(w in cats_text for w in [
        "history", "war", "battle", "revolution", "empire", "dynasty",
        "ancient", "medieval", "century", "politics", "election",
        "government", "democracy", "republic", "parliament", "congress",
        "colonialism", "civilization", "military", "treaty", "independence",
        "political", "nationalism", "socialism", "communism"
    ]):
        return "History & Politics"

    elif any(w in cats_text for w in [
        "geography", "country", "city", "town", "river", "mountain",
        "island", "lake", "ocean", "continent", "region", "district",
        "province", "municipality", "populated places", "settlements",
        "states", "counties", "capitals", "administrative"
    ]):
        return "Geography"

    elif any(w in cats_text for w in [
        "birth", "death", "people", "born", "died", "biography",
        "politicians", "scientists", "artists", "musicians", "actors",
        "writers", "philosophers", "mathematicians", "engineers",
        "presidents", "kings", "queens", "monarchs", "generals"
    ]):
        return "Biography"

    elif any(w in cats_text for w in [
        "film", "music", "album", "song", "band", "art", "novel",
        "book", "literature", "poetry", "television", "sport",
        "football", "cricket", "olympic", "award", "culture",
        "religion", "philosophy", "mythology", "entertainment",
        "theatre", "dance", "painting", "sculpture", "architecture"
    ]):
        return "Culture & Arts"

    else:
        return "General"


def extract_features_and_topics(filename, max_articles=MAX_ARTICLES_PER_CHUNK):
    articles = []
    pages_seen = 0

    print(f"Processing {filename}...")

    with bz2.open(filename, "rb") as f:
        dump = mwxml.Dump.from_file(f)

        for page in dump:
            for revision in page:
                text = revision.text or ""

                if text.strip().startswith("#REDIRECT"):
                    break

                parsed = mwparserfromhell.parse(text)

                categories = [
                    str(link.title)
                    for link in parsed.filter_wikilinks()
                    if str(link.title).startswith("Category:")
                ]
                topic = assign_category_from_cats(categories)

                articles.append({
                    "title":      page.title,
                    "length":     len(text),
                    "refs":       text.count("<ref"),
                    "images":     text.count("[[File:") + text.count("[[Image:"),
                    "sections":   text.count("=="),
                    "links":      len(parsed.filter_wikilinks()),
                    "infobox":    1 if "{{Infobox" in text else 0,
                    "categories": "|".join(categories[:10]),
                    "topic":      topic,
                })
                break

            pages_seen += 1
            if pages_seen >= max_articles:
                break

    print(f"  Done -- {len(articles)} articles extracted ({pages_seen - len(articles)} redirects skipped)")
    return articles


all_articles = []
for chunk in CHUNKS:
    all_articles.extend(extract_features_and_topics(chunk))

df_topics = pd.DataFrame(all_articles).drop_duplicates(subset="title")

print(f"\nTotal articles extracted: {len(df_topics)}")
print("\nTopic distribution:")
print(df_topics["topic"].value_counts())

df_topics.to_csv("wikipedia_with_topics.csv", index=False)
print("\nSaved to wikipedia_with_topics.csv")

## Section 3 : Fetch real quality labels from Wikipedia Talk pages

In [ ]:
def get_real_quality_label(title):
    try:
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "titles": "Talk:" + title,
            "prop": "revisions",
            "rvprop": "content",
            "rvslots": "main",
            "format": "json",
        }
        r = requests.get(url, params=params, headers=HEADERS, timeout=10)
        data = r.json()
        page = list(data["query"]["pages"].values())[0]

        if "revisions" not in page:
            return "Unknown"

        content = page["revisions"][0]["slots"]["main"]["*"].lower()

        if "currentstatus=fa" in content or "|class=fa" in content or "| class = fa" in content:
            return "FA"
        elif "currentstatus=ga" in content or "|class=ga" in content or "| class = ga" in content:
            return "GA"
        elif "|class=b" in content or "| class = b" in content:
            return "B"
        elif "|class=c" in content or "| class = c" in content:
            return "C"
        elif "|class=start" in content or "| class = start" in content:
            return "Start"
        elif "|class=stub" in content or "| class = stub" in content:
            return "Stub"
        else:
            return "Unknown"

    except Exception:
        return "Unknown"


df_topics = pd.read_csv("wikipedia_with_topics.csv")

real_labels = []
total = len(df_topics)
print(f"Fetching real labels for {total} articles...")

for i, title in enumerate(df_topics["title"]):
    real_labels.append(get_real_quality_label(title))
    if (i + 1) % 100 == 0:
        print(f"  Progress: {i + 1}/{total} articles done...")
    time.sleep(0.5)  # be polite to Wikipedia's servers

df_topics["quality_real"] = real_labels
df_topics.to_csv("wikipedia_features_labelled.csv", index=False)

print("\nDone! Saved to wikipedia_features_labelled.csv")
print("\nReal quality label distribution:")
print(df_topics["quality_real"].value_counts())
unknown_count = (df_topics["quality_real"] == "Unknown").sum()
print(f"\nUnknown count: {unknown_count}")

## Section 4 : Clean the dataset and build the final labelled file

In [ ]:
df = pd.read_csv("wikipedia_features_labelled.csv")

unknown = df[df["quality_real"] == "Unknown"]
print(f"Dropping {len(unknown)} articles with no recognisable quality label")

df_final = df[df["quality_real"] != "Unknown"].copy().reset_index(drop=True)


def collapse_quality(q):
    if q in ["Stub", "Start"]:
        return "Low"
    elif q in ["C", "B"]:
        return "Medium"
    else:  # GA, FA
        return "High"


df_final["quality_3class"] = df_final["quality_real"].apply(collapse_quality)

print(f"\nFinal dataset: {len(df_final)} articles")
print("\nSix-class distribution:")
print(df_final["quality_real"].value_counts())
print("\nThree-class distribution:")
print(df_final["quality_3class"].value_counts())
print("\nTopic distribution:")
print(df_final["topic"].value_counts())

df_final.to_csv("wikipedia_final.csv", index=False)
print("\nSaved to wikipedia_final.csv -- ready for Wikipedia_analysis_final.ipynb")